In [1]:
import os
import datetime
import time
from google.oauth2.credentials import Credentials
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from google_auth_oauthlib.flow import InstalledAppFlow
from google.auth.transport.requests import Request
import pytz
from google.oauth2 import service_account
import pandas as pd  # Import pandas

# --- Configuration ---
MAIN_SUBMISSION_FOLDER_ID = '1lv9WOP7a7JIOzNU8DsTER8w9SvdM6Wik'  # Replace with the ID of your main submission folder
CSV_FILE = 'team_data.csv'  # Replace with the name of your CSV file
ACCESS_LEVEL = 'writer'
START_TIME_STR = '2025-05-20 12:00 AM'
END_TIME_STR = '2025-05-20 01:50 PM'
CREDENTIALS_FILE = 'C:/Users/chand/Desktop/NumpyNinja/Access_Control_Automation/credentials.json'
SCOPES = ['https://www.googleapis.com/auth/drive']  # Broader scope for child file permissions

# --- Global Variable to Store Team Folder IDs ---
TEAM_FOLDER_IDS_MAP = {}


def authenticate_oauth():
    creds = None
    if os.path.exists('token.json'):
        creds = Credentials.from_authorized_user_file('token.json', SCOPES)
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file(CREDENTIALS_FILE, SCOPES)
            creds = flow.run_local_server(port=0)
        with open('token.json', 'w') as token:
            token.write(creds.to_json())
    return creds


def get_existing_permissions(drive_service, file_id, email_address):
    try:
        permissions = (
            drive_service.permissions()
            .list(fileId=file_id, fields='permissions(id, emailAddress, role)')
            .execute()
        )
        for perm in permissions.get('permissions', []):
            if perm.get('emailAddress') == email_address:
                return perm.get('id'), perm.get('role')
        return None, None
    except HttpError as error:
        print(f'An error occurred: {error}')
        return None, None


def grant_access_to_team_folder(
    drive_service, folder_id, participant_emails, access_level
):
    """Grants the specified access level to the list of participants for a given folder."""
    for email in participant_emails:
        try:
            permission_id, existing_role = get_existing_permissions(
                drive_service, folder_id, email
            )
            if permission_id:
                if existing_role != access_level:
                    permission = (
                        drive_service.permissions()
                        .update(
                            fileId=folder_id,
                            permissionId=permission_id,
                            body={'role': access_level},
                        )
                        .execute()
                    )
                    print(
                        f'Updated {access_level} access for {email} on folder:'
                        f' {folder_id} (ID: {permission_id})'
                    )
                else:
                    print(
                        f'{email} already has {access_level} access to folder:'
                        f' {folder_id}'
                    )
            else:
                permission = (
                    drive_service.permissions()
                    .create(
                        fileId=folder_id,
                        body={'role': access_level, 'type': 'user', 'emailAddress': email},
                        fields='id',
                    )
                    .execute()
                )
                print(
                    f'Granted {access_level} access to {email} for folder: {folder_id}'
                    f' (ID: {permission["id"]})'
                )
        except HttpError as error:
            print(
                f'An error occurred granting access to {email} for folder {folder_id}:'
                f' {error}'
            )

# DEFINITION OF revoke_access_from_team_folder
def revoke_access_from_team_folder(drive_service, folder_id, participant_emails):
    """Revokes access for the list of participants from a given folder."""
    for email in participant_emails:
        try:
            permission_id, _ = get_existing_permissions(drive_service, folder_id, email)
            if permission_id:
                drive_service.permissions().delete(fileId=folder_id, permissionId=permission_id).execute()
                print(f'Revoked access for {email} from folder: {folder_id} (ID: {permission_id})')
            else:
                print(f'Warning: No existing permission found for {email} on folder: {folder_id}')
        except HttpError as error:
            print(f'An error occurred revoking access for {email} from folder {folder_id}: {error}')

# DEFINITION OF revoke_file_access_from_folder
def revoke_file_access_from_folder(drive_service, folder_id, participant_emails):
    try:
        # List all files within the folder
        results = drive_service.files().list(q=f"'{folder_id}' in parents and mimeType != 'application/vnd.google-apps.folder'", fields="files(id)").execute()
        files = results.get('files', [])

        for file in files:
            file_id = file['id']
            for email in participant_emails:
                try:
                    permission_id, _ = get_existing_permissions(drive_service, file_id, email)
                    if permission_id:
                        drive_service.permissions().delete(fileId=file_id, permissionId=permission_id).execute()
                        print(f"Revoked access from {email} for file: {file_id} in folder: {folder_id}")
                except HttpError as error:
                    print(f"Error revoking access for {email} on file {file_id}: {error}")
    except HttpError as error:
        print(f"Error listing files in folder {folder_id}: {error}")


def find_or_create_team_folders(drive_service, main_folder_id, teams_config):
    global TEAM_FOLDER_IDS_MAP
    if TEAM_FOLDER_IDS_MAP:
        return True  # Folders already created/found

    try:
        # List existing folders in the main submission folder
        results = (
            drive_service.files()
            .list(
                q=f"'{main_folder_id}' in parents and mimeType ="
                " 'application/vnd.google-apps.folder'",
                fields='files(id, name)',
            )
            .execute()
        )
        existing_folders = {f['name']: f['id'] for f in results.get('files', [])}

        for team_name in teams_config:
            if team_name in existing_folders:
                TEAM_FOLDER_IDS_MAP[team_name] = existing_folders[team_name]
                print(
                    f'Found existing folder: {team_name} with ID:'
                    f' {TEAM_FOLDER_IDS_MAP[team_name]}'
                )
            else:
                file_metadata = {
                    'name': team_name,
                    'mimeType': 'application/vnd.google-apps.folder',
                    'parents': [main_folder_id],
                }
                folder = (
                    drive_service.files().create(body=file_metadata, fields='id').execute()
                )
                TEAM_FOLDER_IDS_MAP[team_name] = folder.get('id')
                print(
                    f'Created folder: {team_name} with ID:'
                    f' {TEAM_FOLDER_IDS_MAP[team_name]}'
                )
        return True
    except HttpError as error:  # This is likely around line 100
        print(f'An error occurred finding or creating folders: {error}')
        return False


def main():
    creds = authenticate_oauth()
    try:
        service = build('drive', 'v3', credentials=creds)
        est_timezone = pytz.timezone('America/New_York')
        start_time_naive = datetime.datetime.strptime(
            START_TIME_STR, '%Y-%m-%d %I:%M %p'
        )
        end_time_naive = datetime.datetime.strptime(END_TIME_STR, '%Y-%m-%d %I:%M %p')
        start_time = est_timezone.localize(start_time_naive)
        end_time = est_timezone.localize(end_time_naive)
        now = datetime.datetime.now(est_timezone)

        print(f"Current time (EST): {now.strftime('%Y-%m-%d %I:%M %p %Z%z')}")
        print(f"Start time (EST): {start_time.strftime('%Y-%m-%d %I:%M %p %Z%z')}")
        print(f"End time (EST): {end_time.strftime('%Y-%m-%d %I:%M %p %Z%z')}")

        # Read data from CSV
        print(f"Current working directory: {os.getcwd()}")
        print(f"Checking if CSV file exists: {os.path.exists(CSV_FILE)}")
        try:
            with open(CSV_FILE, 'r') as f:
                first_line = f.readline().strip()
                print(f"First line of CSV file: '{first_line}'")
        except Exception as e:
            print(f"Error reading first line of CSV: {e}")
            return

        TEAMS_CONFIG = {}  # Initialize TEAMS_CONFIG here
        try:
            df = pd.read_csv(CSV_FILE) # Revert to basic read for this test
            print(f"Pandas read CSV successfully. DataFrame info:\n{df.info()}")
            print(f"Pandas read CSV successfully. DataFrame head:\n{df.head()}")
            for index, row in df.iterrows():
                team_number = row['Team Number']
                email = row['Email']
                if team_number not in TEAMS_CONFIG:
                    TEAMS_CONFIG[team_number] = []
                TEAMS_CONFIG[team_number].append(email)
        except FileNotFoundError:
            print(f'Error: CSV file "{CSV_FILE}" not found. Exiting.')
            return
        except KeyError as e:
            print(f'Error: Missing column in CSV: {e}. CSV file must have "Team Number" and "Email" columns. Exiting.')
            return
        except Exception as e:
            print(f"An error occurred while reading the CSV with pandas: {e}")
            return

        print("TEAMS_CONFIG:", TEAMS_CONFIG)
        if not find_or_create_team_folders(
            service, MAIN_SUBMISSION_FOLDER_ID, TEAMS_CONFIG
        ):
            print('Failed to find or create team folders. Exiting.')
            return

        print("TEAM_FOLDER_IDS_MAP:", TEAM_FOLDER_IDS_MAP)

        if start_time <= now < end_time:
            print('Granting access to team folders...')
            for team_name, participant_emails in TEAMS_CONFIG.items():
                folder_id = TEAM_FOLDER_IDS_MAP.get(team_name)
                if folder_id:
                    grant_access_to_team_folder(
                        service, folder_id, participant_emails, ACCESS_LEVEL
                    )
                else:
                    print(f'Warning: Could not find folder ID for team {team_name}')
        elif now >= end_time:
            print('Revoking access from team folders and files...')
            for team_name, participant_emails in TEAMS_CONFIG.items():
                folder_id = TEAM_FOLDER_IDS_MAP.get(team_name)
                if folder_id:
                    revoke_access_from_team_folder(service, folder_id, participant_emails)
                    revoke_file_access_from_folder(service, folder_id, participant_emails)
                else:
                    print(f'Warning: Could not find folder ID for team {team_name}')
        else:
            wait_seconds = (start_time - now).total_seconds()
            print(f'Waiting {wait_seconds:.0f} seconds until start time...')

    except HttpError as error:
        print(f'An error occurred: {error}')


if __name__ == '__main__':
    est_timezone = pytz.timezone('America/New_York')
    while True:
        now = datetime.datetime.now(est_timezone)
        start_time_naive = datetime.datetime.strptime(
            START_TIME_STR, '%Y-%m-%d %I:%M %p'
        )
        end_time_naive = datetime.datetime.strptime(END_TIME_STR, '%Y-%m-%d %I:%M %p')
        start_time = est_timezone.localize(start_time_naive)
        end_time = est_timezone.localize(end_time_naive)

        main()

        if now < start_time:
            wait_seconds = (start_time - now).total_seconds()
            time.sleep(wait_seconds + 5)
        elif start_time <= now < end_time:
            time.sleep(60)
        elif now >= end_time:
            print('Access revocation time reached. Stopping script.')
            break
        time.sleep(5)

Current time (EST): 2025-05-20 01:48 PM EDT-0400
Start time (EST): 2025-05-20 12:00 AM EDT-0400
End time (EST): 2025-05-20 01:50 PM EDT-0400
Current working directory: C:\Users\chand
Checking if CSV file exists: True
First line of CSV file: 'ï»¿Team Number,Email'
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Team Number  10 non-null     object
 1   Email        10 non-null     object
dtypes: object(2)
memory usage: 292.0+ bytes
Pandas read CSV successfully. DataFrame info:
None
Pandas read CSV successfully. DataFrame head:
  Team Number                       Email
0      Team 1  satyasruthithota@gmail.com
1      Team 2    kalyankranthim@gmail.com
2      Team 3     garine.manasa@gmail.com
3      Team 4        niveditayp@gmail.com
4      Team 5   mohanmayaonline@gmail.com
TEAMS_CONFIG: {'Team 1': ['satyasruthithota@gmail.com'], 'Team 2': ['kalyankr